# Experimento 7 — SVM com 5 variáveis, 3 classificações e balanceamento em nova amostra

Este experimento tem como objetivo replicar a melhor configuração obtida na trilha de testes com SVM, agora utilizando uma nova amostra rotulada: `amostra_rotulada_3.parquet`.

Nos experimentos anteriores, o SVM apresentou melhor equilíbrio geral quando foi utilizado com balanceamento de classes. Por isso, neste experimento será mantido o parâmetro `class_weight="balanced"`, que ajuda o modelo a dar mais importância às classes minoritárias.

A principal diferença deste experimento é que a nova amostra trabalha com 3 classificações:

- `Adequada`
- `Atenção`
- `Crítica`

Essa redução torna o problema mais direto, agrupando situações de melhor qualidade hídrica em uma classe adequada e mantendo separadas as situações de atenção e crítica.

O objetivo principal é verificar se o SVM consegue manter bom desempenho em uma nova amostra, avaliando principalmente:

- accuracy de treino e teste;
- diferença entre treino e teste;
- precision por classe;
- recall por classe;
- F1-score por classe;
- matriz de confusão;
- desempenho da classe `Crítica`.

Neste contexto ambiental, a classe `Crítica` é especialmente importante, pois representa situações de maior risco. Assim, o modelo não deve ser avaliado apenas pela acurácia geral, mas também pela capacidade de detectar corretamente amostras críticas.

## Preparação do ambiente

Nesta etapa serão importadas as bibliotecas principais utilizadas no notebook.

Também será definida a semente aleatória (`SEED = 42`) para garantir reprodutibilidade dos resultados.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


## Carregamento da nova amostra

Nesta etapa será carregado o arquivo `amostra_rotulada_3.parquet`.

Essa nova amostra já contém a variável alvo `conama_status` com 3 classes:

- `Adequada`
- `Atenção`
- `Crítica`

O uso dessa nova amostra permite avaliar a robustez do modelo SVM em um cenário diferente dos experimentos anteriores.

In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_3.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path(
        "C:/Users/anaju/OneDrive/projetos_lucas\AquaSense-ML/amostra_rotulada_3.parquet"
    )

df = pd.read_parquet(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente local/VS Code detectado.
Dataset carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Adequada
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Adequada
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Adequada


## Preparação para Machine Learning

Nesta etapa serão importadas as bibliotecas necessárias para:

- separar treino e teste;
- criar o pré-processamento;
- codificar variáveis categóricas;
- padronizar variáveis numéricas;
- treinar o modelo SVM;
- calcular métricas de avaliação.

Será utilizado o `LinearSVC`, pois ele é mais adequado para datasets grandes do que o `SVC` tradicional.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("Bibliotecas de Machine Learning carregadas.")

Bibliotecas de Machine Learning carregadas.


## Definição das variáveis de entrada e saída

Neste experimento serão utilizadas 5 variáveis, seguindo a configuração da nova etapa experimental:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Country`
- `Waterbody Type`
- `Nitrogen (mg/l)`

A variável alvo será:

- `conama_status`

Essa configuração permite avaliar o SVM em um cenário mais simples do que os experimentos com todas as variáveis, mas ainda mantendo informações ambientais relevantes.

In [4]:
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type",
        "Nitrogen (mg/l)"
    ]
]

y = df["conama_status"]

print("X e y definidos.")
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

print("\nDistribuição das classes:")
print(y.value_counts())

print("\nDistribuição percentual das classes:")
print(y.value_counts(normalize=True).round(4))

X e y definidos.
Shape de X: (141399, 5)
Shape de y: (141399,)

Distribuição das classes:
conama_status
Adequada    103469
Atenção      36547
Crítica       1383
Name: count, dtype: int64

Distribuição percentual das classes:
conama_status
Adequada    0.7318
Atenção     0.2585
Crítica     0.0098
Name: proportion, dtype: float64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição das classes no treino:")
print(y_train.value_counts(normalize=True).round(4))

print("\nDistribuição das classes no teste:")
print(y_test.value_counts(normalize=True).round(4))

Treino: (113119, 5)
Teste: (28280, 5)

Distribuição das classes no treino:
conama_status
Adequada    0.7318
Atenção     0.2585
Crítica     0.0098
Name: proportion, dtype: float64

Distribuição das classes no teste:
conama_status
Adequada    0.7318
Atenção     0.2585
Crítica     0.0098
Name: proportion, dtype: float64


## Pré-processamento dos dados

O SVM é sensível à escala das variáveis numéricas. Por isso, as variáveis numéricas serão padronizadas com `StandardScaler`.

As variáveis categóricas serão transformadas com `OneHotEncoder`.

Neste experimento:

Variáveis categóricas:
- `Country`
- `Waterbody Type`

Variáveis numéricas:
- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Nitrogen (mg/l)`

O pré-processamento será incluído dentro do pipeline para evitar vazamento de dados entre treino e teste.

In [6]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Nitrogen (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

print("Pré-processamento configurado.")

Pré-processamento configurado.


## Construção do modelo SVM com balanceamento

Nesta etapa será construído o modelo SVM utilizando `LinearSVC`.

O parâmetro `class_weight="balanced"` será mantido porque a base continua desbalanceada, com predominância da classe `Adequada`.

Esse balanceamento faz com que o modelo atribua maior peso às classes menos frequentes durante o treinamento.

No contexto de qualidade da água, essa estratégia é importante porque a classe `Crítica`, embora tenha poucas amostras, possui grande relevância ambiental.

In [7]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LinearSVC(
                class_weight="balanced",
                random_state=SEED,
                max_iter=5000
            )
        )
    ]
)

print("Treinando SVM com 5 variáveis, 3 classes e balanceamento...")

model.fit(X_train, y_train)

print("Treinamento concluído.")

Treinando SVM com 5 variáveis, 3 classes e balanceamento...
Treinamento concluído.


## Avaliação das métricas de treino

Nesta etapa serão calculadas as métricas no conjunto de treino.

A avaliação em treino permite observar se o modelo aprendeu padrões consistentes ou se há sinais de comportamento instável.

Também será usada posteriormente para calcular a diferença entre treino e teste, ajudando a observar possível overfitting.

In [8]:
y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("\nTrain Precision:")
print(train_precision)

print("\nTrain Recall:")
print(train_recall)

print("\nTrain F1:")
print(train_f1)

print("\nTrain Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.7657069104217682

Train Precision:
0.7571522677308777

Train Recall:
0.7657069104217682

Train F1:
0.7605440616893968

Train Confusion Matrix:
[[72164 10337   274]
 [14105 14354   779]
 [  109   899    98]]


## Avaliação das métricas de teste

Nesta etapa será realizada a avaliação principal do Experimento 7.

O modelo será testado em dados que não foram utilizados durante o treinamento.

A análise deverá considerar:

- acurácia;
- precision;
- recall;
- F1-score;
- matriz de confusão;
- diferença entre treino e teste.

A classe `Crítica` será analisada com atenção especial, pois representa o cenário mais grave no contexto ambiental.

In [9]:
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

test_precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:")
print(test_accuracy)

print("\nDiferença treino/teste (overfitting):")
print(round(train_accuracy - test_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7657001414427157

Diferença treino/teste (overfitting):
0.0

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.83      0.87      0.85     20694
     Atenção       0.56      0.49      0.52      7309
     Crítica       0.07      0.08      0.08       277

    accuracy                           0.77     28280
   macro avg       0.49      0.48      0.48     28280
weighted avg       0.76      0.77      0.76     28280


Confusion Matrix:
[[18066  2560    68]
 [ 3551  3567   191]
 [   31   225    21]]


# Resultados Obtidos e Considerações Finais — Experimento 7

## Replicação do melhor modelo SVM em nova amostra ambiental

O Experimento 7 teve como objetivo validar a robustez do melhor modelo obtido anteriormente na trilha SVM, replicando a estratégia do Experimento 4 em uma nova amostra ambiental.

Diferente dos experimentos anteriores, esta nova etapa utilizou:
- uma nova amostra rotulada (`amostra_rotulada_3.parquet`);
- três classificações ambientais;
- cinco variáveis ambientais;
- e o modelo SVM linear com balanceamento automático de classes.

O objetivo principal foi avaliar se o modelo manteria desempenho estável mesmo quando aplicado em dados diferentes daqueles utilizados anteriormente.



## Resultado geral do modelo

A accuracy obtida no conjunto de teste foi:

```python
0.7657
```

A diferença entre treino e teste foi praticamente nula:

```python
0.0
```

Isso indica excelente estabilidade entre treinamento e generalização, sem sinais relevantes de overfitting.



## Análise das métricas

### Classe Adequada

A classe `Adequada` apresentou excelente desempenho:

- precision: 0.83
- recall: 0.87
- F1-score: 0.85

O modelo conseguiu identificar corretamente a maior parte das amostras de boa qualidade ambiental.

O alto recall demonstra que poucas amostras adequadas foram classificadas incorretamente.

O alto F1-score também mostra bom equilíbrio entre precision e recall nessa classe.



### Classe Atenção

A classe `Atenção` apresentou desempenho intermediário:

- precision: 0.56
- recall: 0.49
- F1-score: 0.52

Os resultados mostram que o modelo conseguiu detectar parcialmente situações intermediárias de qualidade da água.

Entretanto, ainda houve confusão significativa entre:
- `Adequada`;
- e `Atenção`.

Isso sugere que os limites ambientais entre essas categorias continuam relativamente próximos no espaço de atributos utilizado.

Mesmo assim, o desempenho da classe `Atenção` foi razoável considerando o forte desbalanceamento da base.



### Classe Crítica

A classe `Crítica` continuou sendo o maior desafio do modelo:

- precision: 0.07
- recall: 0.08
- F1-score: 0.08

Apesar do uso de `class_weight="balanced"`, o modelo apresentou dificuldade para detectar corretamente situações críticas.

O recall de 0.08 indica que apenas pequena parte das amostras críticas foi identificada corretamente.

Esse comportamento já havia sido observado em experimentos anteriores e demonstra uma limitação estrutural do SVM linear para modelar padrões extremamente raros e complexos no contexto ambiental.



## Análise da matriz de confusão

A matriz de confusão mostra que o modelo classificou corretamente:

- 18066 amostras da classe `Adequada`;
- 3567 amostras da classe `Atenção`;
- 21 amostras da classe `Crítica`.

A maior parte dos erros ocorreu entre:
- `Adequada` e `Atenção`;
- `Atenção` e `Crítica`.

A classe `Crítica` apresentou poucas previsões corretas, além de ser frequentemente confundida com `Atenção`.

Esse comportamento indica que:
- as regiões críticas possuem poucos exemplos;
- os padrões ambientais críticos continuam difíceis de separar linearmente;
- e o modelo prioriza estabilidade global em vez de sensibilidade extrema.



## Análise de estabilidade do modelo

Um dos pontos mais positivos do Experimento 7 foi a estabilidade.

A diferença praticamente nula entre treino e teste mostra que:
- o modelo generalizou bem;
- não houve forte memorização da base;
- e o pipeline permaneceu consistente mesmo em uma nova amostra ambiental.

Isso é extremamente importante em aplicações reais de monitoramento ambiental, onde os modelos precisam funcionar em cenários diferentes daqueles usados durante o treinamento.



## Comparação com os experimentos anteriores do SVM

O Experimento 7 apresentou:
- desempenho global inferior ao Experimento 3 em accuracy;
- desempenho inferior ao Experimento 4 em equilíbrio geral;
- desempenho inferior ao Experimento 5 em recall crítico.

Entretanto, o Experimento 7 possui uma característica diferente:
- ele funciona como validação externa do modelo.

Enquanto os experimentos anteriores avaliavam desempenho dentro da amostra original, o Experimento 7 mede robustez em novos dados.

Nesse contexto, o resultado foi positivo:
- o modelo manteve estabilidade;
- preservou desempenho razoável nas classes principais;
- e não apresentou overfitting significativo.



## Conclusão Final do Experimento 7

O Experimento 7 confirmou que o SVM linear balanceado possui boa capacidade de generalização em novas amostras ambientais.

O modelo apresentou:
- estabilidade entre treino e teste;
- boa performance na classe `Adequada`;
- desempenho razoável na classe `Atenção`;
- e baixo desempenho na classe `Crítica`.

Os resultados reforçam uma conclusão observada ao longo de toda a trilha experimental:
- o SVM é um modelo estável e consistente;
- porém apresenta dificuldade para detectar padrões críticos raros em datasets ambientais altamente desbalanceados.

Mesmo com:
- balanceamento;
- novas variáveis;
- nova amostra;
- e diferentes estratégias experimentais;

a classe `Crítica` continuou sendo o principal desafio.

Ainda assim, o Experimento 7 foi extremamente importante porque demonstrou que o pipeline construído:
- é robusto;
- possui boa capacidade de generalização;
- e mantém comportamento consistente fora da amostra original.

Essa etapa fortalece significativamente a validade metodológica da trilha experimental do AquaSense.